In [1]:
import numpy as np
import tritonclient.grpc as grpcclient
from tritonclient.grpc import InferInput, InferRequestedOutput


In [2]:
def predict_triton(texts, model_name="text_classifier_ensemble", url="localhost:8001"):
    """Отправляет запросы к модели через gRPC"""
    
    # Создаем клиент
    client = grpcclient.InferenceServerClient(url=url)
    
    # Проверяем готовность модели
    if not client.is_model_ready(model_name):
        print(f"Модель {model_name} не готова")
        return None
    
    batch_size = len(texts)
    input_tensor = InferInput("TEXT", [batch_size, 1], "BYTES")

    text_data = np.array([[text.encode('utf-8')] for text in texts], dtype=np.object_)
    input_tensor.set_data_from_numpy(text_data)
    
    # Запрашиваем выходные данные
    output_tensor = InferRequestedOutput("OUTPUT")
    
    # Отправляем запрос
    response = client.infer(
        model_name=model_name,
        inputs=[input_tensor],
        outputs=[output_tensor]
    )
    
    # Получаем результаты: форма [batch_size, 1]
    predictions = response.as_numpy("OUTPUT")
    return predictions


In [3]:
def list_models(url="localhost:8001"):
    """Выводит список доступных моделей"""
    try:
        client = grpcclient.InferenceServerClient(url=url)
        models = client.get_model_repository_index()
        print("Доступные модели:")
        for model in models.models:
            status = "✅ ready" if client.is_model_ready(model.name) else "❌ not ready"
            print(f"   • {model.name} - {status}")
    except Exception as e:
        print(f"Ошибка: {e}")

list_models()

Доступные модели:
   • bert_classifier - ✅ ready
   • text_classifier_ensemble - ✅ ready
   • text_tokenizer - ✅ ready


In [4]:
def sigmoid(x):
    """Сигмоида для преобразования logits в вероятность"""
    return 1 / (1 + np.exp(-x))

In [5]:
# Определение тестовых данных
test_texts = [
    "Хорошее место. могу рекомендовать",
    "Хуже места не встечал!",
    "Это место топ 1 с конца по качеству",
    "Капец как плохо в этом заведении",
    "Капец как хорошо в этом заведении",
]

In [6]:
predictions = predict_triton(test_texts)

# Вывод результатов
if predictions is not None:

    for i, (text, pred) in enumerate(zip(test_texts, predictions)):
        # Получаем logit (выход модели до сигмоиды)
        logit = float(pred[0]) if isinstance(pred, np.ndarray) else float(pred)
        
        # Применяем сигмоиду для получения вероятности
        probability = sigmoid(logit)
        
        if probability > 0.5:
            sentiment = "отрицательный отзыв"
            emoji = "❌"
        else:
            sentiment = "положительный отзыв"
            emoji = "✅"
        
        print(f"\n{emoji} Текст {i+1}:")
        print(f"   \"{text}\"")
        print(f"   Logit: {logit:.4f}")
        print(f"   Вероятность (sigmoid): {probability:.4f}")
        print(f"   {sentiment}")
    


✅ Текст 1:
   "Хорошее место. могу рекомендовать"
   Logit: -1.9460
   Вероятность (sigmoid): 0.1250
   положительный отзыв

❌ Текст 2:
   "Хуже места не встечал!"
   Logit: 1.5778
   Вероятность (sigmoid): 0.8289
   отрицательный отзыв

✅ Текст 3:
   "Это место топ 1 с конца по качеству"
   Logit: -1.7938
   Вероятность (sigmoid): 0.1426
   положительный отзыв

❌ Текст 4:
   "Капец как плохо в этом заведении"
   Logit: 1.6095
   Вероятность (sigmoid): 0.8333
   отрицательный отзыв

✅ Текст 5:
   "Капец как хорошо в этом заведении"
   Logit: -0.2179
   Вероятность (sigmoid): 0.4457
   положительный отзыв


In [7]:
import onnxruntime as ort
from transformers import AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained("triton/models/text_tokenizer/1/tokenizer")
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

def predict_text_onnx(onnx_file_path, tokenizer, text_list, device, max_length=64):
    """
    ONNX inference with preserving torch pipeline
    """
    session = ort.InferenceSession(onnx_file_path, providers=['CPUExecutionProvider'])
    
    inputs = tokenizer(
        text_list,
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors='pt' 
    )
    
    ort_inputs = {
        'input_ids': inputs['input_ids'].cpu().numpy(),
        'attention_mask': inputs['attention_mask'].cpu().numpy()
    }
    
    ort_outputs = session.run(None, ort_inputs)
    
    logits = torch.from_numpy(ort_outputs[0]).to(device)
    probabilities = torch.sigmoid(logits)
    
    return logits, probabilities

text = ['Привет это тестовый текст для проверки onnx модели']

logits, probs = predict_text_onnx("triton/models/bert_classifier/1/model.onnx", tokenizer, test_texts, device, max_length=64)

print("Логиты (PyTorch Tensor):\n", logits)
print("Вероятности после Сигмоиды (PyTorch Tensor):\n", probs)

/home/mordrud/.local/share/mise/installs/python/3.11.15/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Логиты (PyTorch Tensor):
 tensor([[-1.9460],
        [ 1.5782],
        [-1.7937],
        [ 1.6093],
        [-0.2182]], device='cuda:0')
Вероятности после Сигмоиды (PyTorch Tensor):
 tensor([[0.1250],
        [0.8289],
        [0.1426],
        [0.8333],
        [0.4457]], device='cuda:0')
